# DATA 144 Kaggle Competition - Predict survival on the Titanic

Authors: Maryan Le,

## Checkpoint 1 (due 10/7)
The baseline public score (0.74850) is produced by gender_submission.csv. 

Your assignment is to beat the public score of that algorithm. In other words, the first half of this lab will entail the initial construction of your algorithm and your optimization of that algorithm to predict survival better than one that only accounts for gender.

## Checkpoint 1 Submission
You will submit the following:

Kaggle: Upload the .csv file with your predictions and confirm that you appear on the leaderboard.

Gradescope: Submit a screenshot of your team's name, rank, and % accuracy from the leaderboard to Gradescope. The assignment will be called "Lab 5a: Baseline Checkpoint". Include ALL members of your group in Gradescope. While one person needs to submit for your whole group, it is vital that you include all members in your group in Gradescope.

## Reminders
You need to submit as a team.

Make sure your submission has 418 rows.

You need to beat the Baseline Prediction to get credit.

Each team is allowed 5 submissions per day. Days refresh at 5:00pm PT (Kaggle's rules, not ours).
You have 2 weeks to complete the lab but must submit at least 1 prediction and finalize group membership before Wednesday, October 7th at 12:00, to receive full credit. Teams without a submission will receive a 0 on Lab 5a.
Please note that DSP accommodations cannot be applied to this lab, as this is a synchronous competition that works actively on a timeline.

See Evaluation for grading specifics.

# Data Frame Schema

Data Dictionary Survived : 

Survival, 0 = No, 1 = Yes

Pclass : Ticket class, 1 = 1st, 2 = 2nd, 3 = 3rd

Sex : Sex

Age: Age in years

SibSp: Number of siblings / spouses aboard the Titanic

Parch: Number of parents / children aboard the Titanic

Ticket: Ticket number

Fare: Passenger fare

Cabin: Cabin number

Embarked: Part of Embarkation, C = Cherbourg, Q = Queenstown, S = Southampton

Variable Notes Pclass: A proxy for socio-economic status (SES) 1st = Upper 2nd = Middle 3rd = Lower

Age: Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

SibSp: The number of siblings/spouses

Parch: The number of parents/children

Some children travelled only with a nanny, therefore parch=0 for them.

Upload any packages

The coding packages were taken from lab 4

In [57]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_extraction import DictVectorizer

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, ParameterGrid


Load in the test, train, and gender_submission csv files as panda DataFrames

**gender_submission** a set of predictions that assume only female passengers survive, baseline public score is 0.74850

In [12]:
df_test = pd.read_csv("data/test.csv")
df_train = pd.read_csv("data/train.csv")
df_gender = pd.read_csv("data/gender_submission.csv")


In [13]:
df_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [14]:
df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [15]:
df_gender.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


Create a model that will predict whether or not the passenger will survive

**reshape** is used because le returns a 1D array, and we want a 2D array

reshape(-1, 1), -1 is a value that automatically find the number of rows, 1 is the number of columns

each value will be its own row

**ravel** is used because 


## Steps

Fill NA values in the feature age with the median

Label encode the column "Sex" to have 0s and 1s

Extract all the other features from the table and turn them to arrays using .values

stack all the arrays together

Scale the data

Then train your model

In [111]:
# x training is height feature from training
# do one hot encoding on the sex
# create a label encoder for the sex, will do 0 and 1 for male and female
le = LabelEncoder()
le.fit(df_train["Sex"])

sex_train = le.transform(df_train["Sex"]).reshape(-1, 1)
sex_test = le.transform(df_test["Sex"]).reshape(-1, 1)

age_median = df_train["Age"].median()
age_train = df_train["Age"].fillna(age_median).values.reshape(-1, 1)
age_test = df_test["Age"].fillna(age_median).values.reshape(-1, 1)

# sib_train = df_train["SibSp"].values.reshape(-1, 1)
# sib_test = df_test["SibSp"].values.reshape(-1, 1)

# parch_train = df_train["Parch"].values.reshape(-1, 1)
# parch_test = df_test["Parch"].values.reshape(-1, 1)


x_train = np.hstack([sex_train, age_train, sib_train, parch_train])
x_test = np.hstack([sex_test, age_test, sib_test, parch_test])

# scale your data, scaler goes through each column one by one
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train) # FIT AND TRANSFORM
X_test_scaled = scaler.transform(x_test) # ONLY TRANSFORM

# encode gender as a binary class
y_train = df_train['Survived']

mlp = MLPClassifier(hidden_layer_sizes=(30,),
                    activation = 'relu',
                    solver = 'adam',
                    random_state = 42,
                    max_iter = 50
                    )

mlp.fit(X_train_scaled, y_train)

y_train_pred = mlp.predict(X_train_scaled)
y_test_pred = mlp.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, y_train_pred)

print(f"this is the train accuracy {train_accuracy}")

this is the train accuracy 0.8024691358024691


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Convert the predictions into a csv file for the kaggle submission

In [112]:
# turn the model's prediction into a dataframe
df = pd.DataFrame(
    {'PassengerId': df_test["PassengerId"].to_numpy(),
    'Survived': y_test_pred
    }
)

df.head()

# convert to a csv file, without the index column
df.to_csv('predictions.csv', index = False)